# Sesten makine arıza tespiti — Colab'da çalıştırma

GPU'su olmayan ekip üyesi için. Önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seçin.

Akış: projeyi al → paketleri kur → testler → veriyi indir → hattı çalıştır → sonuçları indir.
Colab oturumu kapanınca dosyalar silinir; en sonda `checkpoints` ve `results` klasörlerini indirin veya Drive'a kopyalayın.

## 1. Projeyi al
GitHub deponuz varsa `git clone` satırını kullanın; yoksa proje zip'ini yükleyin.

In [ ]:
# Seçenek A: GitHub
# !git clone https://github.com/<kullanici>/<depo>.git fan-anomaly

# Seçenek B: zip yükle
from google.colab import files
up = files.upload()            # fan-anomaly.zip seçin
!unzip -q -o fan-anomaly.zip

In [ ]:
%cd /content/fan-anomaly
!pip install -q librosa soundfile pyyaml streamlit
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Testler (kod doğru mu?)

In [ ]:
!python -m pytest -q

## 3. Veri
Zenodo'dan `dev_data_fan.zip` indirilir ve `data/raw/fan` altına açılır.

In [ ]:
!python scripts/download_data.py

## 4. Hat
Önce tek kimlik prototipi; sorunsuzsa tüm kimlikler. `--seeds 42 43 44` ile tohum tekrarı.

In [ ]:
!python run_pipeline.py --machine-ids id_00

In [ ]:
# !python run_pipeline.py
# !python run_pipeline.py --seeds 42 43 44

## 5. Sonuçlar

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/summary.md", encoding="utf-8").read()))
import os
if os.path.exists("results/noise_summary.md"):
    display(Markdown(open("results/noise_summary.md", encoding="utf-8").read()))
import glob
for p in sorted(glob.glob("results/figures/roc_*.png")) + ["results/figures/noise_robustness.png"]:
    if os.path.exists(p): display(Image(p))

## 6. Çıktıları indir
`data/split.csv` dosyasını da saklayın — bölünme sabit kalmalı.

In [ ]:
!zip -q -r outputs.zip checkpoints results data/split.csv data/manifest.csv
files.download("outputs.zip")